In [1]:
#!/usr/bin/env python3
"""
Complete Sales Report Processing Workflow
Extracts data from sales report images and converts to structured CSV format
"""

import os
import sys
from pathlib import Path
import pandas as pd
from datetime import datetime

from sales_report import SalesReportExtractor
from coordinate_calibration_tool import CoordinateCalibrator
from quality_check import DataValidator, ManualReviewInterface

class SalesReportProcessor:
    def __init__(self, config_file='processing_config.json'):
        self.config_file = config_file
        self.extractor = SalesReportExtractor()
        self.calibrator = CoordinateCalibrator()
        self.validator = DataValidator()
        
        # Load or create configuration
        self.load_config()
    
    def load_config(self):
        """Load processing configuration"""
        try:
            import json
            with open(self.config_file, 'r') as f:
                self.config = json.load(f)
        except FileNotFoundError:
            # Default configuration
            self.config = {
                'coordinates_calibrated': False,
                'output_directory': 'output',
                'backup_directory': 'backup',
                'quality_threshold': 0.95  # 95% accuracy required
            }
            self.save_config()
    
    def save_config(self):
        """Save configuration to file"""
        import json
        with open(self.config_file, 'w') as f:
            json.dump(self.config, f, indent=2)
    
    def setup_directories(self):
        """Create necessary directories"""
        directories = [
            self.config['output_directory'],
            self.config['backup_directory'],
            'logs',
            'temp'
        ]
        
        for directory in directories:
            Path(directory).mkdir(exist_ok=True)
    
    def calibrate_coordinates(self, sample_image_path):
        """Calibrate extraction coordinates using sample image"""
        print("Starting coordinate calibration...")
        print("This is a one-time setup process.")
        
        coordinates = self.calibrator.calibrate_regions(sample_image_path)
        
        if coordinates:
            # Update extractor with calibrated coordinates
            self.extractor.regions = self.convert_coordinates_format(coordinates)
            self.config['coordinates_calibrated'] = True
            self.save_config()
            print("Coordinate calibration completed!")
            return True
        else:
            print("Coordinate calibration failed!")
            return False
    
    def convert_coordinates_format(self, coordinates):
        """Convert calibrator coordinates to extractor format"""
        regions = {}
        for region_name, coords in coordinates.items():
            regions[region_name] = (
                coords['x1'], coords['y1'], 
                coords['x2'], coords['y2']
            )
        return regions
    
    def process_images(self, image_paths, output_filename=None):
        """Process multiple images and extract data"""
        if not output_filename:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            output_filename = f"extracted_sales_data_{timestamp}.csv"
        
        output_path = os.path.join(self.config['output_directory'], output_filename)
        
        print(f"Processing {len(image_paths)} images...")
        
        # Extract data
        all_data = []
        failed_images = []
        
        for i, img_path in enumerate(image_paths, 1):
            print(f"Processing image {i}/{len(image_paths)}: {os.path.basename(img_path)}")
            
            try:
                data = self.extractor.process_image(img_path)
                all_data.extend(data)
                print(f"  ✓ Extracted {len(data)} transactions")
            except Exception as e:
                print(f"  ✗ Failed: {str(e)}")
                failed_images.append(img_path)
        
        if not all_data:
            print("No data extracted from any images!")
            return None
        
        # Create DataFrame
        df = pd.DataFrame(all_data)
        
        # Save raw extracted data
        raw_output = output_path.replace('.csv', '_raw.csv')
        df.to_csv(raw_output, index=False)
        print(f"Raw data saved to: {raw_output}")
        
        return df, failed_images
    
    def validate_and_review(self, df):
        """Validate extracted data and handle manual review"""
        print("\nValidating extracted data...")
        
        # Run validation
        validation_results = self.validator.validate_dataset(df)
        
        # Generate validation report
        report_path = os.path.join(self.config['output_directory'], 'validation_report.txt')
        self.validator.generate_validation_report(validation_results, report_path)
        
        # Calculate accuracy
        accuracy = validation_results['valid_rows'] / validation_results['total_rows']
        print(f"Data accuracy: {accuracy:.1%}")
        print(f"Valid rows: {validation_results['valid_rows']}/{validation_results['total_rows']}")
        
        # Apply automatic fixes
        print("Applying automatic fixes...")
        df_fixed = self.validator.fix_common_issues(df)
        
        # Re-validate after fixes
        validation_results_fixed = self.validator.validate_dataset(df_fixed)
        accuracy_fixed = validation_results_fixed['valid_rows'] / validation_results_fixed['total_rows']
        print(f"Accuracy after automatic fixes: {accuracy_fixed:.1%}")
        
        # Manual review if accuracy is below threshold
        if accuracy_fixed < self.config['quality_threshold']:
            print(f"\nAccuracy ({accuracy_fixed:.1%}) is below threshold ({self.config['quality_threshold']:.1%})")
            
            if validation_results_fixed['rows_with_errors'] > 0:
                review = input("Start manual review? (y/n): ").lower().strip()
                
                if review == 'y':
                    reviewer = ManualReviewInterface(df_fixed, validation_results_fixed)
                    df_final = reviewer.review_errors()
                    
                    # Final validation
                    final_results = self.validator.validate_dataset(df_final)
                    final_accuracy = final_results['valid_rows'] / final_results['total_rows']
                    print(f"Final accuracy: {final_accuracy:.1%}")
                    
                    return df_final
        
        return df_fixed
    
    def run_full_workflow(self, image_paths, sample_image=None):
        """Run the complete processing workflow"""
        print("SALES REPORT PROCESSING WORKFLOW")
        print("=" * 50)
        
        # Setup
        self.setup_directories()
        
        # Coordinate calibration (if needed)
        if not self.config['coordinates_calibrated']:
            if not sample_image:
                sample_image = image_paths[0] if image_paths else None
            
            if sample_image:
                if not self.calibrate_coordinates(sample_image):
                    print("Calibration failed. Cannot proceed.")
                    return None
            else:
                print("No sample image provided for calibration!")
                return None
        
        # Process images
        result = self.process_images(image_paths)
        if result is None:
            return None
        
        df, failed_images = result
        
        # Report failed images
        if failed_images:
            print(f"\nFailed to process {len(failed_images)} images:")
            for img in failed_images:
                print(f"  - {img}")
        
        # Validate and review
        df_final = self.validate_and_review(df)
        
        # Save final output
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        final_output = os.path.join(
            self.config['output_directory'], 
            f"final_sales_data_{timestamp}.csv"
        )
        df_final.to_csv(final_output, index=False)
        
        print(f"\n🎉 Processing complete!")
        print(f"Final data saved to: {final_output}")
        print(f"Total transactions: {len(df_final)}")
        
        return df_final, final_output

def main():
    """Main entry point"""
    processor = SalesReportProcessor()
    
    # Get image files from command line or ask user
    if len(sys.argv) > 1:
        image_paths = sys.argv[1:]
    else:
        print("Enter image file paths (one per line, empty line to finish):")
        image_paths = []
        while True:
            path = input().strip()
            if not path:
                break
            if os.path.exists(path):
                image_paths.append(path)
            else:
                print(f"File not found: {path}")
    
    if not image_paths:
        print("No valid image paths provided!")
        return
    
    # Run workflow
    result = processor.run_full_workflow(image_paths)
    
    if result:
        df, output_file = result
        print(f"\nSUCCESS: {len(df)} transactions extracted to {output_file}")
    else:
        print("FAILED: Could not process images")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'sales_report'